In [43]:
import pandas as pd
import optuna
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os
import shutil
from huggingface_hub import hf_hub_download

# 1. Load Dataset & Preprocessing

In [44]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

data_path = 'dataset_model/dataset_trainable_rev2.csv'

In [45]:
if os.path.exists(data_path):
    print(f"Dataset Raw lokal sudah ada di '{data_path}'.")
else:
    print(f"Dataset Raw lokal belum ada. Mengunduh dataset dari Hugging Face...")
    
    temp_downloaded_path = hf_hub_download(
        repo_id="vel740/dhaka-smart-waste-processed",
        filename="processed/dataset_trainable_rev2.csv",
        repo_type="dataset"
    )
    
    os.makedirs('dataset_model', exist_ok=True)
    shutil.copy(temp_downloaded_path, data_path)
    print(f"Dataset selesai diunduh dan tersimpan di '{data_path}'.")

df = pd.read_csv(data_path)
print(f"Total baris awal: {len(df):,} baris")

Dataset Raw lokal sudah ada di 'dataset_model/dataset_trainable_rev2.csv'.
Total baris awal: 15,650 baris


In [46]:
target_diff = df['target_next_hour'] - df['fill_level_percent']
velocity_3h = df['lag_2h'] - df['lag_3h']

clean_mask = (
    (target_diff > -20.0) & 
    (df['velocity_1h'] > -20.0) & 
    (df['velocity_2h'] > -20.0) & 
    (velocity_3h > -20.0)
)

df_clean = df[clean_mask].reset_index(drop=True)
print(f"Total baris setelah pembersihan event pickup: {len(df_clean):,} baris (Dibuang: {len(df) - len(df_clean)} baris)")

Total baris setelah pembersihan event pickup: 15,650 baris (Dibuang: 0 baris)


In [47]:
df.head(5)

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,lag_24h,lag_48h,is_weekend,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1735840800,67.709990,68.51879,66.02875,67.10285,42.061382,14.483168,0,-0.808800,2.490044,0.000000,1.000000,-0.433884,-0.900969,70.403190
1,0,1735844400,70.403190,67.70999,68.51879,66.02875,42.492905,15.378024,0,2.693199,-0.808800,0.258819,0.965926,-0.433884,-0.900969,69.584000
2,0,1735848000,69.584000,70.40319,67.70999,68.51879,44.316532,16.866007,0,-0.819191,2.693199,0.500000,0.866025,-0.433884,-0.900969,72.692300
3,0,1735851600,72.692300,69.58400,70.40319,67.70999,43.484684,18.462954,0,3.108299,-0.819191,0.707107,0.707107,-0.433884,-0.900969,68.450325
4,0,1735855200,68.450325,72.69230,69.58400,70.40319,44.515327,20.921236,0,-4.241974,3.108299,0.866025,0.500000,-0.433884,-0.900969,72.475070


In [48]:
df_clean.head(5)

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,lag_24h,lag_48h,is_weekend,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1735840800,67.709990,68.51879,66.02875,67.10285,42.061382,14.483168,0,-0.808800,2.490044,0.000000,1.000000,-0.433884,-0.900969,70.403190
1,0,1735844400,70.403190,67.70999,68.51879,66.02875,42.492905,15.378024,0,2.693199,-0.808800,0.258819,0.965926,-0.433884,-0.900969,69.584000
2,0,1735848000,69.584000,70.40319,67.70999,68.51879,44.316532,16.866007,0,-0.819191,2.693199,0.500000,0.866025,-0.433884,-0.900969,72.692300
3,0,1735851600,72.692300,69.58400,70.40319,67.70999,43.484684,18.462954,0,3.108299,-0.819191,0.707107,0.707107,-0.433884,-0.900969,68.450325
4,0,1735855200,68.450325,72.69230,69.58400,70.40319,44.515327,20.921236,0,-4.241974,3.108299,0.866025,0.500000,-0.433884,-0.900969,72.475070


# 2. Train-Test Split

In [49]:
features = [
    'fill_level_percent', 'lag_1h', 'lag_2h', 'lag_3h', 
    'lag_24h', 'lag_48h', 'is_weekend',
    'velocity_1h', 'velocity_2h', 
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]
target = 'target_next_hour'

split_timestamp = df_clean['timestamp_unix'].quantile(0.8)

scaler = StandardScaler()
train_mask_global = df_clean['timestamp_unix'] <= split_timestamp
scaler.fit(df_clean.loc[train_mask_global, features])
train_mask = df_clean['timestamp_unix'] <= split_timestamp
test_mask = df_clean['timestamp_unix'] > split_timestamp

X_train_raw = df_clean.loc[train_mask, features].values
y_train_raw = df_clean.loc[train_mask, target].values
X_test_raw = df_clean.loc[test_mask, features].values
y_test_raw = df_clean.loc[test_mask, target].values

In [50]:
df_scaled = df_clean.copy()
df_scaled[features] = scaler.transform(df_clean[features])

In [ ]:
def create_sequences_grouped(df, seq_length, feature_cols, target_col, split_time):
    X_tr, y_tr, X_te, y_te = [], [], [], []
    
    for bin_id, group in df.groupby('bin_index'):
        X_val = group[feature_cols].values
        y_val = group[target_col].values
        time_val = group['timestamp_unix'].values
        
        for i in range(len(group) - seq_length):
            seq_x = X_val[i:(i + seq_length)]
            seq_y = y_val[i + seq_length]
            target_time = time_val[i + seq_length]
            

            if target_time <= split_time:
                X_tr.append(seq_x)
                y_tr.append(seq_y)
            else:
                X_te.append(seq_x)
                y_te.append(seq_y)
                
    return np.array(X_tr), np.array(y_tr), np.array(X_te), np.array(y_te)
SEQUENCE_LENGTH = 12 
X_train_seq, y_train_seq, X_test_seq, y_test_seq = create_sequences_grouped(
    df_scaled, SEQUENCE_LENGTH, features, target, split_timestamp
)

In [52]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

train_dataset = TensorDataset(torch.tensor(X_train_seq, dtype=torch.float32), 
                              torch.tensor(y_train_seq, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test_seq, dtype=torch.float32), 
                             torch.tensor(y_test_seq, dtype=torch.float32))

Training on device: cuda


In [53]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False) 
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 3. Baseline Model

In [54]:
class SmartBinLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(SmartBinLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :]) 
        return out

In [55]:
model = SmartBinLSTM(input_size=len(features), hidden_size=32, num_layers=1, output_size=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

EPOCHS = 50

print("\n=== STARTING PYTORCH LSTM TRAINING ===")
for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()         
        predictions = model(X_batch)  
        
        loss = criterion(predictions.view(-1), y_batch.view(-1))
        loss.backward()             
        optimizer.step()              
        
        train_losses.append(loss.item())
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] - MSE Loss: {np.mean(train_losses):.4f}")


=== STARTING PYTORCH LSTM TRAINING ===
Epoch [10/50] - MSE Loss: 64.9842
Epoch [20/50] - MSE Loss: 56.4295
Epoch [30/50] - MSE Loss: 55.1428
Epoch [40/50] - MSE Loss: 53.2538
Epoch [50/50] - MSE Loss: 50.3040


In [56]:
model.eval()
all_preds = []
all_targets = []

In [57]:
with torch.no_grad(): 
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        
        all_preds.extend(preds.reshape(-1))
        all_targets.extend(y_batch.numpy().reshape(-1))

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

final_mae = mean_absolute_error(all_targets, all_preds)
final_rmse = np.sqrt(mean_squared_error(all_targets, all_preds))

print("\n=== HASIL EVALUASI BASELINE LSTM ===")
print(f"Overall MAE (Semua Kondisi): {final_mae:.4f}%")
print(f"Overall RMSE               : {final_rmse:.4f}%")
print(f"LightGBM MAE Baseline      : 1.6079%")
print(f"Difference                 : {final_mae - 1.6079:+.4f}%")


=== HASIL EVALUASI BASELINE LSTM ===
Overall MAE (Semua Kondisi): 3.5046%
Overall RMSE               : 8.0791%
LightGBM MAE Baseline      : 1.6079%
Difference                 : +1.8967%


# 4. Model Tuning & Evaluation

In [58]:
class SmartBinLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_rate=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        lstm_drop = dropout_rate if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=lstm_drop)
        self.dropout = nn.Dropout(dropout_rate)   
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

In [59]:
def objective(trial):
    seq_length = trial.suggest_int('seq_length', 3, 12)
    hidden_size = trial.suggest_categorical('hidden_size', [16, 32, 64, 128])
    num_layers = trial.suggest_int('num_layers', 1, 3)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    
    X_tr_seq, y_tr_seq, X_te_seq, y_te_seq = create_sequences_grouped(
        df_scaled, seq_length, features, target, split_timestamp
    )
    
    train_dataset = TensorDataset(torch.tensor(X_tr_seq, dtype=torch.float32), 
                                  torch.tensor(y_tr_seq, dtype=torch.float32))
    test_dataset = TensorDataset(torch.tensor(X_te_seq, dtype=torch.float32), 
                                 torch.tensor(y_te_seq, dtype=torch.float32))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=0)
    
    # Initialize Model
    model = SmartBinLSTM(len(features), hidden_size, num_layers, 1, dropout_rate).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Training Loop (30 Epochs per trial)
    EPOCHS = 30
    model.train()
    for epoch in range(EPOCHS):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(X_batch)

            loss = criterion(preds.view(-1), y_batch.view(-1))
            
            loss.backward()
            optimizer.step()
            
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu().numpy()
            
            all_preds.extend(preds.reshape(-1))
            all_targets.extend(y_batch.numpy().reshape(-1))
            
    return mean_absolute_error(all_targets, all_preds)

In [60]:
study_name = "pytorch_lstm_smartbin_v2"
db_filename = "sqlite:///optuna_smartbin_study.sqlite3"

study = optuna.create_study(
    study_name=study_name,
    storage=db_filename,
    direction='minimize',
    load_if_exists=True
)

print(f"Starting GPU Optuna Tuning on {device}...")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n=== OPTUNA LSTM TUNING COMPLETE ===")
print(f"Best Validation MAE: {study.best_value:.4f}%")
print("Best Parameters:", study.best_params)

Starting GPU Optuna Tuning on cuda...


Best trial: 44. Best value: 3.33049: 100%|██████████| 50/50 [24:39<00:00, 29.59s/it]


=== OPTUNA LSTM TUNING COMPLETE ===
Best Validation MAE: 3.3305%
Best Parameters: {'seq_length': 5, 'hidden_size': 64, 'num_layers': 1, 'dropout_rate': 0.14193709724733977, 'learning_rate': 0.0021768581788238093, 'batch_size': 64}


# 5. Save Model

In [61]:
best_params = study.best_params

print("Hasil Parameter Optimal dijalankan melalui Optuna ===")
print(f"Sequence Length : {best_params['seq_length']}")
print(f"Hidden Size     : {best_params['hidden_size']}")
print(f"Num Layers      : {best_params['num_layers']}")
print(f"Dropout Rate    : {best_params['dropout_rate']:.4f}")
print(f"Learning Rate   : {best_params['learning_rate']:.6f}")
print(f"Batch Size      : {best_params['batch_size']}")

OPTIMAL_SEQ_LENGTH = best_params['seq_length']

X_tr_final, y_tr_final, X_te_final, y_te_final = create_sequences_grouped(
    df_scaled, OPTIMAL_SEQ_LENGTH, features, target, split_timestamp
)

train_dataset_final = TensorDataset(torch.tensor(X_tr_final, dtype=torch.float32), 
                                    torch.tensor(y_tr_final, dtype=torch.float32))

train_loader_final = DataLoader(
    train_dataset_final, 
    batch_size=best_params['batch_size'], 
    shuffle=False,
    pin_memory=True,
    num_workers=0
)

Hasil Parameter Optimal dijalankan melalui Optuna ===
Sequence Length : 5
Hidden Size     : 64
Num Layers      : 1
Dropout Rate    : 0.1419
Learning Rate   : 0.002177
Batch Size      : 64


In [62]:
final_model = SmartBinLSTM(
    len(features),
    best_params['hidden_size'],
    best_params['num_layers'],
    1,
    best_params['dropout_rate'],
).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params['learning_rate'])

In [63]:
EPOCHS = 50
print("Training Final Golden Model on GPU...")
final_model.train()
for epoch in range(EPOCHS):
    for X_batch, y_batch in train_loader_final:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = final_model(X_batch)
        loss = criterion(preds.view(-1), y_batch.view(-1))
        loss.backward()
        optimizer.step()

save_dir = 'model'
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, 'lstm_smartbin_16days_final.pth')

torch.save(final_model.state_dict(), model_path)
print(f"\nSUCCESS: Model weights safely locked and exported to -> {model_path}")

Training Final Golden Model on GPU...

SUCCESS: Model weights safely locked and exported to -> model\lstm_smartbin_16days_final.pth
